# Taller autónomo | Árboles, Bagging, Random Forest y Extra-Trees
**Duración sugerida:** 4 horas; los experimentos extensos pueden completarse fuera de clase.
**Objetivo:** comparar cinco clasificadores con datos propios, mediante AP, AUC-ROC, validación cruzada, OOB y costos de clasificación.

Nombre(s): ... | Fuente y URL del dataset: ...

Seleccione un CSV preferiblemente de 1000 filas y cinco predictores numéricos como mínimo. Debe tener un objetivo binario, suficientes ejemplos de ambas clases y observaciones independientes. No use Wine Quality. Justifique la clase positiva, unidad de análisis, fuente, licencia, variables y utilidad del modelo. Excluya identificadores y variables que revelen el objetivo o información futura. Para este taller básico se emplean predictores numéricos completos; si su conjunto requiere codificación o imputación, impleméntela dentro de un Pipeline por pliegue o seleccione otro conjunto. No elimine duplicados sin comprobar si son registros realmente repetidos. No use el test hasta la evaluación final.

Cada celda tiene una tarea y una explicación. Los espacios TODO deben completarse; las demás celdas son andamios que puede adaptar. No se proporcionan resultados esperados porque cada dataset es diferente.

## 1. Librerías
`pandas` organiza tablas, `numpy` realiza operaciones numéricas, `matplotlib` grafica. Los clasificadores, métricas y herramientas de validación provienen de scikit-learn. La semilla permite reproducir las particiones y el azar de los árboles.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import BaggingClassifier, ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    RepeatedStratifiedKFold,
    StratifiedKFold,
    cross_val_predict,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

SEMILLA = 42
from sklearn.base import clone
from sklearn.metrics import ConfusionMatrixDisplay
from pathlib import Path

## 2. Importar su propio CSV (una sola importación)
En Colab ejecute la carga interactiva. En VS Code coloque el archivo junto al notebook y escriba su nombre. Ajuste `sep` y `decimal` si el CSV usa punto y coma o coma decimal. `read_csv` lee el archivo una sola vez.

In [ ]:
import sys
if 'google.colab' in sys.modules:
    from google.colab import files
    archivo = next(iter(files.upload()))
else:
    archivo = input('Ruta del CSV: ')
datos = pd.read_csv(archivo, sep=',', decimal='.')
display(datos.head())
print(datos.shape)
datos.info()

## 3. Diagnóstico y definición del problema
`isna().sum()` cuenta faltantes; `duplicated().sum()` identifica filas repetidas; `describe` resume las variables. Antes de modelar, escriba el significado de cada columna, su unidad y si estará disponible al predecir.

In [ ]:
display(datos.describe(include='all').T)
print('Faltantes:'); print(datos.isna().sum())
print('Filas duplicadas:', datos.duplicated().sum())
print('Columnas:', datos.columns.tolist())

### Preguntas de interpretación (responda debajo)
1. ¿Qué representa una fila?
2. ¿Cuál es la fuente y el problema?
3. ¿Qué variables excluirá y por qué?
4. ¿Los duplicados son errores o eventos legítimos?

**Respuesta del estudiante:** ...

## 4. Configurar X e y
Complete los nombres reales. `map` transforma las dos clases en 0 y 1; `dropna` elimina registros incompletos en las columnas seleccionadas (justifique esta decisión y cuantifique las pérdidas). Si existen otras categorías en el objetivo, el código se detiene: defina explícitamente cómo transformarlas antes de continuar. Seleccione solo predictores numéricos y no use el objetivo como predictor.

In [ ]:
OBJETIVO = '' # TODO: nombre de la variable objetivo
CLASE_POSITIVA = None # TODO: valor original de la clase 1 (texto o número)
CLASE_NEGATIVA = None # TODO: valor original de la clase 0
PREDICTORAS = [] # TODO: lista de nombres de columnas numéricas
ELIMINAR_DUPLICADOS = False # cambie solo con justificación
assert OBJETIVO in datos.columns and PREDICTORAS, 'Complete la configuración'
assert OBJETIVO not in PREDICTORAS and len(set(PREDICTORAS)) == len(PREDICTORAS)
assert CLASE_POSITIVA is not None and CLASE_NEGATIVA is not None and CLASE_POSITIVA != CLASE_NEGATIVA
base = datos.drop_duplicates() if ELIMINAR_DUPLICADOS else datos.copy()
base = base[PREDICTORAS + [OBJETIVO]].dropna().copy()
assert base[OBJETIVO].isin([CLASE_NEGATIVA, CLASE_POSITIVA]).all(), 'Objetivo debe tener exactamente las dos clases definidas'
X = base[PREDICTORAS].apply(pd.to_numeric, errors='raise')
y = base[OBJETIVO].map({CLASE_NEGATIVA:0, CLASE_POSITIVA:1}).astype(int)
assert np.isfinite(X.to_numpy()).all(), 'Revise infinitos'
assert y.value_counts().reindex([0,1],fill_value=0).min() >= 10, 'Se necesitan más ejemplos de ambas clases'
print('Filas conservadas:', len(base), 'de', len(datos))
display(y.value_counts().sort_index())
print('Prevalencia positiva:', y.mean())
y.value_counts().sort_index().plot.bar(title='Distribución de clases'); plt.show()

### Preguntas de interpretación (responda debajo)
1. ¿Qué porcentaje pertenece a la clase positiva?
2. ¿Existe desbalance?
3. ¿Qué pérdidas de datos produjo la limpieza y qué sesgo podría introducir?

**Respuesta del estudiante:** ...

## 5. Partición y modelo de referencia
`train_test_split` reserva 20% para prueba. `stratify=y` conserva aproximadamente la proporción de clases. `StratifiedKFold` divide exclusivamente el entrenamiento en cinco pliegues. `Pipeline` ajusta el escalado dentro de cada pliegue, evitando fuga de información. AP resume la curva precisión-recall y debe compararse con la prevalencia; AUC-ROC mide la capacidad de ordenar positivos frente a negativos. El error estándar aquí es descriptivo, no un intervalo formal de confianza.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEMILLA)
assert y_train.value_counts().min() >= 5 and y_test.nunique()==2
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
print('Entrenamiento:', X_train.shape, 'Prueba:', X_test.shape)
def evaluar_cv(modelo, X=X_train, y=y_train, cv=cv):
    """AP y AUC-ROC promedio de la CV, con error estándar y tiempo de ajuste por pliegue."""
    r = cross_validate(modelo, X, y, cv=cv, scoring=["average_precision", "roc_auc"])
    k = len(r["test_average_precision"])
    return pd.Series(
        {
            "AP": r["test_average_precision"].mean(),
            "ee AP": r["test_average_precision"].std(ddof=1) / np.sqrt(k),
            "AUC-ROC": r["test_roc_auc"].mean(),
            "ajuste (s)": r["fit_time"].mean(),
        }
    )


logistica = Pipeline([("escalar", StandardScaler()), ("clf", LogisticRegression(max_iter=2000))])
referencia = evaluar_cv(logistica)
print("Regresión logística:")
print(referencia.round(3).to_string())

### Preguntas de interpretación (responda debajo)
1. Interprete AP, AUC-ROC y tiempo.
2. ¿Por qué no se escala antes de dividir los datos?
3. ¿Qué diferencia existe entre AP y accuracy?

**Respuesta del estudiante:** ...

## 6. Árbol de decisión: profundidad
Pruebe los diez valores de `max_depth` del notebook original. `None` permite crecer sin límite explícito. Cada punto representa la AP media de cinco pliegues; las barras representan un error estándar. Compare con la línea de logística. Seleccione la profundidad por AP de CV, no por el test.

In [ ]:
profundidades = [1, 2, 3, 4, 5, 6, 8, 10, 15, None]
curva_arbol = pd.DataFrame(
    {str(d): evaluar_cv(DecisionTreeClassifier(max_depth=d, random_state=SEMILLA)) for d in profundidades}
).T

plt.figure(figsize=(7, 4))
plt.errorbar(range(len(profundidades)), curva_arbol["AP"], yerr=curva_arbol["ee AP"], fmt="o-", capsize=3, label="árbol")
plt.axhline(referencia["AP"], color="gray", ls="--", label=f"logística ({referencia['AP']:.3f})")
plt.xticks(range(len(profundidades)), [str(d) for d in profundidades])
plt.xlabel("max_depth")
plt.ylabel("AP (validación cruzada)")
plt.title("Comparación de profundidad frente a logística")
plt.legend()
plt.show()

mejor_prof = curva_arbol["AP"].idxmax()
print(f"Mejor profundidad: {mejor_prof} → AP = {curva_arbol.loc[mejor_prof, 'AP']:.3f} ± {curva_arbol.loc[mejor_prof, 'ee AP']:.3f}")
print(f"Sin límite:        AP = {curva_arbol.loc['None', 'AP']:.3f}")

### Preguntas de interpretación (responda debajo)
1. ¿Cuál profundidad obtuvo mayor AP?
2. ¿Un árbol sin límite mejora o empeora?
3. ¿Hay evidencia compatible con sobreajuste? ¿Qué evidencia adicional necesitaría?

**Respuesta del estudiante:** ...

## 7. Bagging y OOB
`BaggingClassifier` entrena árboles sobre muestras bootstrap. `oob_decision_function_` contiene predicciones de los árboles que no usaron cada registro. Se excluyen filas sin cobertura OOB antes de calcular AP. Use 10, 30, 100 y 300 árboles; 1000 es una extensión opcional de mayor costo computacional.

In [ ]:
Bs = [10, 30, 100, 300]
filas = []
for B in Bs:
    bag = BaggingClassifier(DecisionTreeClassifier(), n_estimators=B, oob_score=True, random_state=SEMILLA, n_jobs=-1)
    bag.fit(X_train, y_train)
    p_oob = bag.oob_decision_function_[:, 1]
    con_oob = ~np.isnan(p_oob)  # filas que al menos un árbol dejó fuera
    ap_oob = average_precision_score(y_train[con_oob], p_oob[con_oob])
    r = evaluar_cv(bag)
    filas.append({"B": B, "AP (CV)": r["AP"], "ee": r["ee AP"], "AP (OOB)": ap_oob,
                  "filas sin OOB": int((~con_oob).sum())})
curva_bagging = pd.DataFrame(filas).set_index("B")
print(curva_bagging.round(3).to_string())
curva_bagging[["AP (CV)", "AP (OOB)"]].plot(marker="o"); plt.ylabel("AP"); plt.show()

### Preguntas de interpretación (responda debajo)
1. ¿Desde cuántos árboles se estabiliza AP?
2. Compare OOB con CV.
3. ¿Por qué hay filas sin OOB con pocos árboles?
4. Justifique el número elegido.

**Respuesta del estudiante:** ...

## 8. Random Forest: hiperparámetros
`max_features` controla cuántos predictores se consideran por división y `min_samples_leaf` el mínimo de registros en cada hoja. La rejilla se adapta al número real de columnas. Para reducir tiempo se usan 100 árboles por combinación; opcionalmente repita con 300, como en la guía. Interprete la tabla por filas y columnas.

In [ ]:
rejilla_mf = sorted({1, min(2,X.shape[1]), min(3,X.shape[1]), min(6,X.shape[1]), X.shape[1]})
rejilla_msl = [1, 2, 5, 10]
resultados_rf = pd.DataFrame(index=rejilla_mf, columns=rejilla_msl, dtype=float)
ee_rf = resultados_rf.copy()
for mf in rejilla_mf:
    for msl in rejilla_msl:
        r = evaluar_cv(RandomForestClassifier(n_estimators=100, max_features=mf, min_samples_leaf=msl, random_state=SEMILLA, n_jobs=-1))
        resultados_rf.loc[mf, msl] = r["AP"]
        ee_rf.loc[mf, msl] = r["ee AP"]

resultados_rf.index.name = "max_features"
resultados_rf.columns.name = "min_samples_leaf"
print("AP de validación cruzada (filas: max_features; columnas: min_samples_leaf):")
print(resultados_rf.round(3).to_string())
print(f"\nError estándar típico: {ee_rf.values.mean():.3f}")
mejor_mf, mejor_msl = resultados_rf.stack().idxmax()
print("Mejor combinación:", mejor_mf, mejor_msl)

### Preguntas de interpretación (responda debajo)
1. Identifique la mejor combinación.
2. ¿Qué sucede al considerar todas las variables?
3. ¿Qué efecto tiene incrementar el mínimo de registros por hoja?

**Respuesta del estudiante:** ...

## 9. Comparación de los cinco modelos
Los modelos comparten la misma CV. Use la profundidad elegida y los parámetros RF seleccionados. Para una comparación reproducible, se mantienen 300 estimadores en los tres ensambles. Extra-Trees añade aleatoriedad en umbrales de división y no usa bootstrap por defecto. Ordene por AP y examine AUC y tiempo. Las puntuaciones de selección pueden ser optimistas; el test sigue reservado.

In [ ]:
prof = None if mejor_prof == 'None' else int(mejor_prof)
candidatos = {
 'Logística': logistica,
 'Árbol': DecisionTreeClassifier(max_depth=prof, random_state=SEMILLA),
 'Bagging': BaggingClassifier(DecisionTreeClassifier(), n_estimators=300, random_state=SEMILLA, n_jobs=-1),
 'Random Forest': RandomForestClassifier(n_estimators=300, max_features=int(mejor_mf), min_samples_leaf=int(mejor_msl), random_state=SEMILLA,n_jobs=-1),
 'Extra-Trees': ExtraTreesClassifier(n_estimators=300,random_state=SEMILLA,n_jobs=-1)
}
tabla = pd.DataFrame({n:evaluar_cv(modelo) for n,modelo in candidatos.items()}).T.sort_values('AP',ascending=False)
display(tabla.round(3))
tabla[['AP','AUC-ROC']].plot.bar(); plt.ylim(0,1); plt.show()

### Preguntas de interpretación (responda debajo)
1. ¿Qué modelo tiene mayor AP y cuál mayor AUC?
2. ¿Coinciden?
3. ¿Cuál es el costo computacional relativo?
4. ¿Qué modelo elegiría provisionalmente y por qué?

**Respuesta del estudiante:** ...

## 10. Comparación pareada
Se utilizan los mismos 20 pliegues (5 × 4) para comparar modelos. La diferencia positiva favorece al primero. `std(ddof=1)/sqrt(n)` es una medida descriptiva: los pliegues no son independientes, por lo que no demuestra significación estadística. Este bloque puede ser el más lento.

In [ ]:
cv_rep = RepeatedStratifiedKFold(n_splits=5,n_repeats=4,random_state=SEMILLA)
ap_pliegues = {n:cross_validate(candidatos[n],X_train,y_train,cv=cv_rep,scoring='average_precision')['test_score'] for n in ['Logística','Bagging','Random Forest','Extra-Trees']}
def comparar(a,b):
    d = ap_pliegues[a]-ap_pliegues[b]
    ee = d.std(ddof=1)/np.sqrt(len(d))
    return pd.Series({'diferencia media':d.mean(),'ee':ee,'cociente':d.mean()/ee if ee>0 else np.nan})
comparaciones = pd.DataFrame({'RF - Logística':comparar('Random Forest','Logística'),'RF - Bagging':comparar('Random Forest','Bagging'),'Extra-Trees - RF':comparar('Extra-Trees','Random Forest')}).T
display(comparaciones.round(3))

### Preguntas de interpretación (responda debajo)
1. Interprete el signo y magnitud de cada diferencia.
2. ¿Por qué se comparan los mismos pliegues?
3. ¿La diferencia justifica el costo adicional?

**Respuesta del estudiante:** ...

## 11. Umbral y costos de clasificación
Defina costos relativos positivos de FP y FN según su contexto. Por ejemplo, si un FN cuesta tres veces un FP, use 1 y 3; no presente estos valores como costos reales sin evidencia. Las probabilidades fuera de pliegue se obtienen mediante `cross_val_predict`. Evalúe 91 umbrales y elija el menor costo exclusivamente en entrenamiento. Compare el ganador por AP con la logística; si gana logística, compare con el segundo modelo.

In [ ]:
COSTO_FP = None # TODO: costo relativo positivo
COSTO_FN = None # TODO: costo relativo positivo
assert COSTO_FP is not None and COSTO_FN is not None and COSTO_FP>0 and COSTO_FN>0
umbrales = np.round(np.linspace(.05,.95,91),2)
def costo_total(y,p,umbral):
    pred = p>=umbral
    return COSTO_FP*np.sum(pred & (y==0))+COSTO_FN*np.sum(~pred & (y==1))
nombre = tabla.index[0]
comparador = 'Logística' if nombre!='Logística' else tabla.index[1]
seleccion = [nombre,comparador]
probas = {n:cross_val_predict(candidatos[n],X_train,y_train,cv=cv,method='predict_proba')[:,1] for n in seleccion}
costos = pd.DataFrame({n:[costo_total(y_train.to_numpy(),p,u) for u in umbrales] for n,p in probas.items()},index=umbrales)
optimos = costos.idxmin()
display(pd.DataFrame({'umbral':optimos,'costo CV':costos.min()}))
costos.plot(); plt.xlabel('Umbral'); plt.ylabel('Costo relativo total'); plt.show()

### Preguntas de interpretación (responda debajo)
1. ¿Qué significa FP y FN en su problema?
2. Justifique la relación de costos.
3. Compare el umbral óptimo con 0.5.
4. ¿Qué modelo escogería considerando costo y AP?

**Respuesta del estudiante:** ...

## 12. Evaluación final en prueba
Ajuste los dos modelos con todo el entrenamiento, aplique sus umbrales ya elegidos y mida AP, AUC, precisión, recall, F1 y costo. La matriz tiene filas reales y columnas predichas. No cambie modelos ni umbrales con estos resultados.

In [ ]:
filas=[]
for n in seleccion:
    modelo=clone(candidatos[n]).fit(X_train,y_train)
    p=modelo.predict_proba(X_test)[:,1]
    for u in sorted({0.5,float(optimos[n])}):
        pred=(p>=u).astype(int)
        vn,fp,fn,vp=confusion_matrix(y_test,pred,labels=[0,1]).ravel()
        filas.append({'modelo':n,'umbral':u,'AP':average_precision_score(y_test,p),'AUC-ROC':roc_auc_score(y_test,p),'precision':precision_score(y_test,pred,zero_division=0),'recall':recall_score(y_test,pred,zero_division=0),'F1':f1_score(y_test,pred,zero_division=0),'VN':vn,'FP':fp,'FN':fn,'VP':vp,'costo':costo_total(y_test.to_numpy(),p,u)})
        ConfusionMatrixDisplay.from_predictions(y_test,pred,labels=[0,1]); plt.title(f'{n} | umbral {u:.2f}'); plt.show()
display(pd.DataFrame(filas).round(3))

### Preguntas de interpretación (responda debajo)
1. Interprete VN, FP, FN y VP en unidades de su problema.
2. ¿Cómo cambian precisión, recall y costo al ajustar el umbral?
3. ¿Qué diferencias aparecen entre validación y prueba?
4. ¿Qué riesgos tendría desplegar este modelo?

**Respuesta del estudiante:** ...

## 13. Conclusiones y entrega
Escriba 250–350 palabras: problema y procedencia, modelo recomendado, evidencia cuantitativa, errores críticos, limitaciones y acciones futuras. Entregue CSV original, notebook ejecutado con respuestas, y URL de fuente. No copie los valores de Wine Quality.

## Rúbrica (100 puntos)
| Criterio | Puntos |
|---|---:|
| Fuente, problema, importación y preparación | 15 |
| Partición, CV y logística | 10 |
| Árbol y profundidad | 10 |
| Bagging y OOB | 15 |
| Random Forest y rejilla | 15 |
| Cinco modelos y comparación pareada | 15 |
| Umbrales, costos y test | 15 |
| Conclusiones y reproducibilidad | 5 |

## Referencias metodológicas (de la guía)
Breiman, L. (1996). Bagging predictors. *Machine Learning, 24*, 123–140. https://doi.org/10.1007/BF00058655

Breiman, L. (2001). Random forests. *Machine Learning, 45*, 5–32. https://doi.org/10.1023/A:1010933404324

Geurts, P., Ernst, D., & Wehenkel, L. (2006). Extremely randomized trees. *Machine Learning, 63*, 3–42. https://doi.org/10.1007/s10994-006-6226-1

Scikit-learn developers. (s. f.). *Ensemble methods*. https://scikit-learn.org/stable/modules/ensemble.html